# Difference-in-Differences (DiD) Template

This notebook demonstrates how to use the improved `did.py` module for Difference-in-Differences analysis, including Two-Way Fixed Effects (TWFE), Event Studies, and **Advanced Techniques** like Stacked DiD and Sensitivity Analysis.

In [ ]:
import sys
import os
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

# Add src to path
sys.path.append(os.path.abspath('../src'))

# Force reload of external modules if they change
%load_ext autoreload
%autoreload 2

from did import (
    simulate_did_panel, did_twfe, event_study, visualize_trends, visualize_event_study,
    simulate_staggered_did, run_stacked_did, run_placebo_test, run_leave_one_out_sensitivity, visualize_sensitivity
)

%matplotlib inline
plt.style.use('ggplot')

## 1. Classic DiD & Event Study

We start with a standard setup where treatment starts at the same time for all treated units.

In [ ]:
df = simulate_did_panel(n_units=200, n_periods=12, treat_start=7, effect=1.5, seed=42)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
visualize_trends(df, y='y', unit='unit', time='time', treated='treated', treat_start=7, ax=ax1)
res_event = event_study(df, y='y', unit='unit', time='time', treated='treated', treat_start=7, cluster='unit')
visualize_event_study(res_event, ax=ax2)
plt.tight_layout()
plt.show()

## 2. Advanced: Staggered Treatment Timing

When units are treated at different times, standard TWFE can be biased. Here we use **Stacked DiD** to avoid these issues.

In [ ]:
# Simulate staggered treatment
df_staggered = simulate_staggered_did(n_units=200, n_periods=15, treat_share=0.5, min_treat_start=5, max_treat_start=10, effect=1.5)

print("Units treated at different times:")
print(df_staggered.groupby('treat_start')['unit'].nunique())

# Run Stacked DiD
res_stacked = run_stacked_did(df_staggered, y='y', unit='unit', time='time', treat_start_col='treat_start', cluster='unit')

print(f"\nStacked DiD Estimated Effect: {res_stacked.params['did']:.4f}")

## 3. Sensitivity Analysis: Placebo Tests

A placebo test randomly re-assigns treatment to see if the found effect is likely to occur by chance.

In [ ]:
print("Running Placebo Test (50 iterations)...")
placebos = run_placebo_test(df, y='y', unit='unit', time='time', treated='treated', post='post', n_iterations=50)

visualize_sensitivity(placebo_effects=placebos, true_effect=1.5, title="Placebo Test")
plt.show()

## 4. Sensitivity Analysis: Leave-One-Out (Jackknife)

Check if the result is driven by a single outlier unit.

In [ ]:
print("Running Leave-One-Out Sensitivity...")
# Note: Running for a subset to save time in example
loo_res = run_leave_one_out_sensitivity(df.iloc[:2000], y='y', unit='unit', time='time', treated='treated', post='post')

visualize_sensitivity(loo_results=loo_res, true_effect=1.5, title="Leave-One-Out Sensitivity")
plt.show()